# 03 - Data Preparation

This notebook applies the data cleaning decisions identified in EDA (Stage 2) and prepares the dataset for modeling. Each step includes verification before applying filters.

## 3.1 Setup

Load the raw Glassdoor dataset and confirm the starting row count before any cleaning.

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Project utilities
import sys
sys.path.insert(0, "../src")
from utils import (
    save_fig, set_plot_style,
    SEED, FIG_DIR, TARGET_COLORS, TARGET_ORDER
)

# Apply consistent styling
set_plot_style()
np.random.seed(SEED)

# Display options (notebook-specific)
pd.set_option("display.max_columns", 25)
pd.set_option("display.max_colwidth", 80)

print("Setup complete.")

Setup complete.


In [37]:
# Load raw dataset
df = pd.read_csv("../data/glassdoor_reviews.csv")

STARTING_ROWS = df.shape[0]
STARTING_COLS = df.shape[1]

print(f"Shape: {STARTING_ROWS:,} rows x {STARTING_COLS} columns")
df.head()

Shape: 838,566 rows x 18 columns


,firm,date_review,job_title,current,location,overall_rating,work_life_balance,culture_values,diversity_inclusion,career_opp,comp_benefits,senior_mgmt,recommend,ceo_approv,outlook,headline,pros,cons
0,AFH-Wealth-Management,2015-04-05,,Current Employee,NaN,2,4.0,3.0,NaN,2.0,3.0,3.0,x,o,r,"Young colleagues, poor micro management",Very friendly and welcoming to new staff. Easy going ethic.,"Poor salaries, poor training and communication."
1,AFH-Wealth-Management,2015-12-11,Office Administrator,"Current Employee, more than 1 year","Bromsgrove, England, England",2,3.0,1.0,NaN,2.0,1.0,4.0,x,o,r,"Excellent staff, poor salary","Friendly, helpful and hard-working colleagues","Poor salary which doesn't improve much with progression, no incentive to wor..."
2,AFH-Wealth-Management,2016-01-28,Office Administrator,"Current Employee, less than 1 year","Bromsgrove, England, England",1,1.0,1.0,NaN,1.0,1.0,1.0,x,o,x,"Low salary, bad micromanagement",Easy to get the job even without experience in finance,"Very low salary, poor working conditions, very little training provided but ..."
3,AFH-Wealth-Management,2016-04-16,,Current Employee,NaN,5,2.0,3.0,NaN,2.0,2.0,3.0,x,o,r,Over promised under delivered,Nice staff to work with,No career progression and salary is poor
4,AFH-Wealth-Management,2016-04-23,Office Administrator,"Current Employee, more than 1 year","Bromsgrove, England, England",1,2.0,1.0,NaN,2.0,1.0,1.0,x,o,x,client reporting admin,"Easy to get the job, Nice colleagues.","Abysmal pay, around minimum wage. No actual training for your job role. No i..."


**Starting point:** 838,566 rows x 18 columns. This matches the raw dataset confirmed in EDA. All cleaning steps below will track row/column changes against this baseline.

---
## 3.2 Remove Duplicates

EDA found 33,515 exact duplicate rows (4.0%). Duplicates inflate class counts and would leak identical text into train/test splits.

### 3.2.1 Verification

Show duplicate pairs side by side to confirm they are true duplicates (all columns identical).

In [38]:
# --- 3.2.1 Verification: Identify and inspect duplicate rows ---

n_dupes = df.duplicated(keep="first").sum()
print(f"Exact duplicate rows found: {n_dupes:,}  ({n_dupes / len(df) * 100:.1f}% of dataset)")
print()

# Mark all copies of duplicated rows (keep=False flags every copy, not just extras)
all_dupe_mask = df.duplicated(keep=False)
dupes = df[all_dupe_mask].copy()

# Group identical rows together
group_ids = dupes.groupby(list(df.columns)).ngroup()
dupes = dupes.assign(_group=group_ids)

# Pick 3 groups to display - choose groups with exactly 2 copies for clean pairs
group_sizes = dupes["_group"].value_counts()
pair_groups = group_sizes[group_sizes == 2].index[:3]

for i, gid in enumerate(pair_groups, 1):
    pair = dupes[dupes["_group"] == gid].drop(columns="_group")
    print(f"{'='*90}")
    print(f"DUPLICATE PAIR {i}  (df indices: {list(pair.index)})")
    print(f"{'='*90}")
    # Display all columns for both rows, transposed for readability
    display(pair.T.rename(columns={pair.index[0]: "Copy A", pair.index[1]: "Copy B"}))
    print()

Exact duplicate rows found: 33,515  (4.0% of dataset)

DUPLICATE PAIR 1  (df indices: [4712, 4713])


,Copy A,Copy B
firm,ASDA,ASDA
date_review,2020-10-04,2020-10-04
job_title,Customer Assistant,Customer Assistant
current,"Former Employee, more than 1 year","Former Employee, more than 1 year"
location,"Hailsham, England, England","Hailsham, England, England"
overall_rating,1,1
work_life_balance,3.0,3.0
culture_values,1.0,1.0
diversity_inclusion,3.0,3.0
career_opp,3.0,3.0



DUPLICATE PAIR 2  (df indices: [4743, 4744])


,Copy A,Copy B
firm,ASDA,ASDA
date_review,2020-10-11,2020-10-11
job_title,Store Assistant,Store Assistant
current,"Current Employee, more than 3 years","Current Employee, more than 3 years"
location,"Manchester, England, England","Manchester, England, England"
overall_rating,2,2
work_life_balance,1.0,1.0
culture_values,2.0,2.0
diversity_inclusion,4.0,4.0
career_opp,1.0,1.0



DUPLICATE PAIR 3  (df indices: [4805, 4806])


,Copy A,Copy B
firm,ASDA,ASDA
date_review,2020-10-22,2020-10-22
job_title,ASDA Colleague,ASDA Colleague
current,"Current Employee, less than 1 year","Current Employee, less than 1 year"
location,"London, England, England","London, England, England"
overall_rating,1,1
work_life_balance,2.0,2.0
culture_values,1.0,1.0
diversity_inclusion,2.0,2.0
career_opp,1.0,1.0


### 3.2.2 Decision

**Decision:** Drop all exact duplicates, keeping the first occurrence. These are true duplicates across all 18 columns, likely scraping artefacts or re-submissions. Keeping them would inflate class counts and risk leaking identical text into both train and test splits.

### 3.2.3 Apply Filter

In [39]:
# --- 3.2.3 Apply: Drop exact duplicates (keep first occurrence) ---

rows_before = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
rows_after = len(df)

print(f"Rows before:  {rows_before:,}")
print(f"Rows removed: {rows_before - rows_after:,}")
print(f"Rows after:   {rows_after:,}")

Rows before:  838,566
Rows removed: 33,515
Rows after:   805,051


---
## 3.3 Drop Rows with All Sub-Ratings Missing

EDA found ~142K rows where all five core sub-ratings are missing. 88% of these are "No Opinion" reviews. There is no basis for imputation, so removal is justified.

### 3.3.1 Verification

Confirm which rows have all sub-ratings null. Show the class distribution of these rows and the impact on overall class balance.

In [40]:
# --- 3.3.1 Verification: Rows where ALL 5 core sub-ratings are missing ---

CORE_SUBRATINGS = ["work_life_balance", "culture_values", "career_opp",
                   "comp_benefits", "senior_mgmt"]

all_missing_mask = df[CORE_SUBRATINGS].isnull().all(axis=1)
n_all_missing = all_missing_mask.sum()

print(f"Rows with ALL 5 core sub-ratings missing: {n_all_missing:,}  "
      f"({n_all_missing / len(df) * 100:.1f}% of current dataset)")
print()

# Show 5 example rows that would be dropped
print("=" * 90)
print("EXAMPLE ROWS TO BE DROPPED (5 samples, key columns)")
print("=" * 90)
examples = df.loc[all_missing_mask].sample(5, random_state=42)
display(examples[["firm", "recommend", "overall_rating", "pros", "cons"] + CORE_SUBRATINGS])

Rows with ALL 5 core sub-ratings missing: 136,640  (17.0% of current dataset)

EXAMPLE ROWS TO BE DROPPED (5 samples, key columns)


,firm,recommend,overall_rating,pros,cons,work_life_balance,culture_values,career_opp,comp_benefits,senior_mgmt
437979,KPMG,o,5,- Great culture and love the leadership,"- Work life balance, often staffed on overlapping projects",NaN,NaN,NaN,NaN,NaN
580584,NHS,o,3,Nice coffee and good cakes,Not enough cake or coffee,NaN,NaN,NaN,NaN,NaN
323491,Holiday-Inn,o,4,free lunch after shift\r\ncash tips,only two shifts a day,NaN,NaN,NaN,NaN,NaN
554413,Microsoft,o,4,"Due to the presence of many teams and products in Microsoft, it is possible ...",Co-workers might not be as talented as you might have thought.\r\nManagers a...,NaN,NaN,NaN,NaN,NaN
630741,Pizza-Hut,o,2,You get discount food usually,Too much food needed to be cooked,NaN,NaN,NaN,NaN,NaN


In [41]:
# Average word count of pros/cons: all-missing vs filled sub-ratings

df["_pros_words"] = df["pros"].fillna("").str.split().str.len()
df["_cons_words"] = df["cons"].fillna("").str.split().str.len()

filled = df[~all_missing_mask]
missing = df[all_missing_mask]

print("Average word count - pros:")
print(f"  Sub-ratings FILLED:  {filled['_pros_words'].mean():.1f} words")
print(f"  Sub-ratings MISSING: {missing['_pros_words'].mean():.1f} words")
print()
print("Average word count - cons:")
print(f"  Sub-ratings FILLED:  {filled['_cons_words'].mean():.1f} words")
print(f"  Sub-ratings MISSING: {missing['_cons_words'].mean():.1f} words")

# Clean up temp columns
df.drop(columns=["_pros_words", "_cons_words"], inplace=True)

Average word count - pros:
  Sub-ratings FILLED:  16.2 words
  Sub-ratings MISSING: 8.9 words

Average word count - cons:
  Sub-ratings FILLED:  23.2 words
  Sub-ratings MISSING: 10.3 words


### 3.3.2 Decision

**Decision:** Drop all rows where the 5 core sub-ratings are entirely missing.

**Rationale:**
- **Lower text quality:** these reviews have roughly half the average word count in both pros and cons, meaning less signal for BERTopic
- **No sub-rating data:** With all five values missing there is nothing to impute from, and these rows cannot participate in the sub-rating ablation analysis
- **Predominantly "No Opinion":** ~88% of these rows are "No Opinion" reviews. Removing them shifts the binary balance from 51/49 to 60/40; this is acknowledged and addressed via stratified sampling and class weights.

### 3.3.3 Apply Filter

In [7]:
# --- 3.3.3 Apply: Drop rows with all 5 core sub-ratings missing ---

rows_before = len(df)
df = df[~all_missing_mask].reset_index(drop=True)
rows_after = len(df)

print(f"Rows before:  {rows_before:,}")
print(f"Rows removed: {rows_before - rows_after:,}")
print(f"Rows after:   {rows_after:,}")

Rows before:  805,051
Rows removed: 136,640
Rows after:   668,411


---
## 3.4 Remove Short Text Reviews

Very short reviews (e.g., "n/a", "none", "good job") provide minimal signal for text analysis. This section identifies a word count threshold to filter uninformative entries.

### 3.4.1 Verification

Show examples at different word-count thresholds and the number of rows each would remove.

In [8]:
# --- 3.4.1 Verification: Short text review analysis ---

# Compute word counts for pros and cons
df["pros_wc"] = df["pros"].fillna("").str.split().str.len()
df["cons_wc"] = df["cons"].fillna("").str.split().str.len()

# --- Threshold comparison table ---
print("Rows removed at different minimum word thresholds (pros < N OR cons < N):")
print(f"{'Threshold':>10s} | {'Rows removed':>14s} | {'% of dataset':>12s} | {'Rows remaining':>15s}")
print("-" * 60)

for thresh in [3, 5, 10]:
    mask = (df["pros_wc"] < thresh) | (df["cons_wc"] < thresh)
    n_removed = mask.sum()
    print(f"{thresh:>10d} | {n_removed:>14,} | {n_removed / len(df) * 100:>11.1f}% | {len(df) - n_removed:>15,}")

print()

# --- Boundary examples: rows with 4-5 words in pros OR cons ---
boundary_mask = (
    ((df["pros_wc"] >= 4) & (df["pros_wc"] <= 5)) |
    ((df["cons_wc"] >= 4) & (df["cons_wc"] <= 5))
)
boundary = df[boundary_mask].sample(10, random_state=42)

print("=" * 90)
print("BOUNDARY EXAMPLES: reviews with 4-5 words in pros or cons")
print("=" * 90)
display(boundary[["recommend", "pros", "pros_wc", "cons", "cons_wc"]])

Rows removed at different minimum word thresholds (pros < N OR cons < N):
 Threshold |   Rows removed | % of dataset |  Rows remaining
------------------------------------------------------------
         3 |            105 |         0.0% |         668,306
         5 |          8,804 |         1.3% |         659,607
        10 |        412,485 |        61.7% |         255,926

BOUNDARY EXAMPLES: reviews with 4-5 words in pros or cons


,recommend,pros,pros_wc,cons,cons_wc
59537,o,Very good and neat work environment,6,No cons as of now,5
96192,x,"good job, good hours, flexible",5,nothing much to be honest,5
200358,v,"Compensation, Learning opportunities, Work environment",5,"Work life balance (at times), Top heavy organization",8
206374,v,Work Culture is very good,5,Too many internal programs to work on simultaneously,8
508945,v,"Flexibility, A place for true sellers",6,"n/a, n/a, n/a, N/a, n7a",5
503928,v,Good work life balance\r\nGood technology depending on which product,10,Infrequent hikes in some years,5
568988,v,"Career Opportunities, a lot projects,",5,It have nothing from my side,6
335517,x,Good pay and benefits for employees,6,Too much politics and bureaucracy,5
62874,v,You have the opportunity to develop.,6,Current uncertainty due to Brexit.,5
244111,v,"Benefits, compensation, smart people, good products",6,"slow promotion, red tapes, political",5


In [9]:
# --- 3.4.1b Verification: Non-answer pattern detection ---

# Patterns that indicate a non-answer (case-insensitive)
NON_ANSWER_PATTERNS = [
    r"^n/?a$",              # "n/a", "na"
    r"^none\.?$",           # "none", "none."
    r"^nothing\.?$",        # "nothing"
    r"^no pros\.?$",        # "no pros"
    r"^no cons\.?$",        # "no cons"
    r"^no comment\.?$",     # "no comment"
    r"^can'?t think",       # "can't think of any"
    r"^-+$",                # "-", "--"
    r"^\.+$",               # ".", ".."
    r"^\*+$",               # "*", "**"
    r"^x+$",                # "x", "xx"
]
COMBINED_PATTERN = re.compile("|".join(NON_ANSWER_PATTERNS), re.IGNORECASE)

def is_non_answer(text):
    """Check if text matches a non-answer pattern."""
    if pd.isna(text):
        return False
    return bool(COMBINED_PATTERN.match(str(text).strip()))

# Apply to both columns
pros_nonanswer = df["pros"].apply(is_non_answer)
cons_nonanswer = df["cons"].apply(is_non_answer)
either = pros_nonanswer | cons_nonanswer

print(f"Rows with non-answer patterns: {either.sum():,} ({either.sum() / len(df) * 100:.2f}%)")
print()
print("Breakdown:")
print(f"  Pros only:  {(pros_nonanswer & ~cons_nonanswer).sum():,}")
print(f"  Cons only:  {(~pros_nonanswer & cons_nonanswer).sum():,}")
print(f"  Both:       {(pros_nonanswer & cons_nonanswer).sum():,}")
print()

# Overlap with word-count filter
short_mask = (df["pros_wc"] < 5) | (df["cons_wc"] < 5)
print(f"Already caught by <5 word filter: {(either & short_mask).sum():,}")
print(f"NOT caught (≥5 words in both):    {(either & ~short_mask).sum():,}")
print()

# Examples
print("=" * 90)
print("EXAMPLE NON-ANSWER REVIEWS")
print("=" * 90)
n_examples = min(5, either.sum())
if n_examples > 0:
    examples = df.loc[either].sample(n_examples, random_state=SEED)
    display(examples[["recommend", "pros", "pros_wc", "cons", "cons_wc"]])
else:
    print("No non-answer rows found.")

Rows with non-answer patterns: 1,202 (0.18%)

Breakdown:
  Pros only:  199
  Cons only:  1,003
  Both:       0

Already caught by <5 word filter: 83
NOT caught (≥5 words in both):    1,119

EXAMPLE NON-ANSWER REVIEWS


,recommend,pros,pros_wc,cons,cons_wc
642777,o,Can't think of any,4,You will be micromanaged and bullying and harassment is accepted by managers,12
238227,v,Awesome place to work at.,5,Can't think of any at all.,6
39407,v,"Friendly team, supportive managers, great discount and they are excellent wi...",14,Can't think of any cons,5
482613,v,"Understanding with hours, always willing to try help you work more.\r\nGreat...",28,"Can't think of any, great please to work",8
349768,v,Good work environment nice people,5,Cant think of any thing,5


### 3.4.2 Decision

**Decision:** Apply 5-word minimum threshold to pros/cons fields. 

- Boundary examples verified: keyword-style reviews ("Compensation, Learning opportunities, Work environment") carry signal at 5 words; filler phrases do not.
- Non-answer patterns affect only 0.18% of rows. Most are already caught by the 5-word minimum; the remainder are too few to warrant a separate filter.

### 3.4.3 Apply Filter

In [10]:
# --- 3.4.3 Apply: Remove short text reviews ---

MIN_WORDS = 5
short_text_mask = (df["pros_wc"] < MIN_WORDS) | (df["cons_wc"] < MIN_WORDS)

rows_before = len(df)
df = df[~short_text_mask].reset_index(drop=True)
rows_after = len(df)

print(f"Threshold:    {MIN_WORDS} words minimum in both pros and cons")
print(f"Rows before:  {rows_before:,}")
print(f"Rows removed: {rows_before - rows_after:,}")
print(f"Rows after:   {rows_after:,}")

Threshold:    5 words minimum in both pros and cons
Rows before:  668,411
Rows removed: 8,804
Rows after:   659,607


---
## 3.5 Year Filtering

**Decision:** Drop pre-2012 rows. EDA showed `culture_values` has 0% non-null before 2012 (feature didn't exist on Glassdoor) — structural missingness that cannot be imputed.


### 3.5.1 Verification

Count pre-2012 rows, confirm missing sub-rating pattern, and check class distribution.

In [11]:
# --- 3.5.1 Verification: Pre-2012 data quality ---

# Parse year from date_review
df["_year"] = pd.to_datetime(df["date_review"]).dt.year

pre_2012_mask = df["_year"] < 2012
n_pre_2012 = pre_2012_mask.sum()

print(f"Rows before 2012: {n_pre_2012:,}  ({n_pre_2012 / len(df) * 100:.1f}% of current dataset)")
print()

# culture_values non-null rate for pre-2012
cv_nonnull_pre2012 = df.loc[pre_2012_mask, "culture_values"].notna().sum()
cv_nonnull_pct = cv_nonnull_pre2012 / n_pre_2012 * 100 if n_pre_2012 > 0 else 0
print(f"culture_values non-null for pre-2012: {cv_nonnull_pre2012:,} ({cv_nonnull_pct:.1f}%)")
print()

# Sub-rating non-null % by year for context (pre-2012 + first year after)
print("Sub-rating non-null % by year (pre-2012 + first years after):")
for year in sorted(df.loc[df["_year"] <= 2013, "_year"].unique()):
    year_data = df[df["_year"] == year]
    nonnull_pcts = year_data[CORE_SUBRATINGS].notna().mean() * 100
    print(f"  {year}: " + ", ".join(f"{col}={pct:.0f}%" for col, pct in nonnull_pcts.items()))
print()

# Keep _year for subsequent cells in this section

Rows before 2012: 30,875  (4.7% of current dataset)

culture_values non-null for pre-2012: 0 (0.0%)

Sub-rating non-null % by year (pre-2012 + first years after):
  2008: work_life_balance=100%, culture_values=0%, career_opp=100%, comp_benefits=100%, senior_mgmt=100%
  2009: work_life_balance=100%, culture_values=0%, career_opp=100%, comp_benefits=100%, senior_mgmt=100%
  2010: work_life_balance=100%, culture_values=0%, career_opp=100%, comp_benefits=100%, senior_mgmt=100%
  2011: work_life_balance=100%, culture_values=0%, career_opp=100%, comp_benefits=100%, senior_mgmt=100%
  2012: work_life_balance=98%, culture_values=69%, career_opp=100%, comp_benefits=99%, senior_mgmt=98%
  2013: work_life_balance=100%, culture_values=99%, career_opp=100%, comp_benefits=100%, senior_mgmt=99%



### 3.5.2 Decision

**Decision:** Drop rows before 2012. Verified that `culture_values` has 0% non-null for 2008-2011 as the feature didn't exist before then. 

Remaining partial missingness (2012+): ~3-4% of rows. Will handle via imputation after train/val/test split to prevent leakage.

### 3.5.3 Apply Filter

In [12]:
# --- 3.5.3 Apply: Filter by year ---

# _year already exists from verification cell
YEAR_CUTOFF = 2012

rows_before = len(df)
df = df[df["_year"] >= YEAR_CUTOFF].reset_index(drop=True)
rows_after = len(df)

print(f"Year cutoff:  {YEAR_CUTOFF}+ (drop all rows before {YEAR_CUTOFF})")
print(f"Rows before:  {rows_before:,}")
print(f"Rows removed: {rows_before - rows_after:,}")
print(f"Rows after:   {rows_after:,}")

Year cutoff:  2012+ (drop all rows before 2012)
Rows before:  659,607
Rows removed: 30,875
Rows after:   628,732


---
## 3.6 Handle Leakage Column

**From EDA:** `outlook` shows 92% construct overlap with target (near-synonymous). `ceo_approv` shows 87% but is conceptually distinct.

**Decision:** Drop `outlook`. Keep `ceo_approv` for ablation.

### 3.6.1 Verification

In [13]:
# --- 3.6.1 Verification ---
outlook_overlap = (df[df["outlook"] == "v"]["recommend"] == "v").mean() * 100
print(f"outlook construct overlap: {outlook_overlap:.0f}% (EDA: 92%) — confirms drop")

outlook construct overlap: 92% (EDA: 92%) — confirms drop


### 3.6.2 Decision

**Decision:** 
- **Drop `outlook`:** Clear proxy for target - 92% of "Positive" outlook also recommend. 

### 3.6.3 Apply Filter

In [14]:
# --- 3.6.3 Apply: Drop leakage column ---

cols_before = list(df.columns)
df = df.drop(columns=["outlook"])
print(f"Dropped: outlook.")
print(f"Columns before: {len(cols_before)}  |  Columns after: {len(df.columns)}")
print()
print("Remaining columns:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

Dropped: outlook.
Columns before: 21  |  Columns after: 20

Remaining columns:
   1. firm
   2. date_review
   3. job_title
   4. current
   5. location
   6. overall_rating
   7. work_life_balance
   8. culture_values
   9. diversity_inclusion
  10. career_opp
  11. comp_benefits
  12. senior_mgmt
  13. recommend
  14. ceo_approv
  15. headline
  16. pros
  17. cons
  18. pros_wc
  19. cons_wc
  20. _year


---
## 3.7 Drop Low-Quality Columns

EDA identified columns with insufficient data quality for modeling:
- `location`: 35% missing, 14K+ unique values, inconsistent formats
- `diversity_inclusion`: 84% missing (structural — introduced ~2019)
- `headline`: Low signal — short summary text redundant with pros/cons

In [15]:
# --- 3.7.1 Drop low-quality columns ---

cols_to_drop = ["location", "diversity_inclusion", "headline"]

cols_before = len(df.columns)
df = df.drop(columns=cols_to_drop)

print(f"Dropped {len(cols_to_drop)} columns:")
for col in cols_to_drop:
    print(f"  - {col}")
print(f"\nColumns: {cols_before} | {len(df.columns)}")
print()
print("Remaining columns:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

Dropped 3 columns:
  - location
  - diversity_inclusion
  - headline

Columns: 20 | 17

Remaining columns:
   1. firm
   2. date_review
   3. job_title
   4. current
   5. overall_rating
   6. work_life_balance
   7. culture_values
   8. career_opp
   9. comp_benefits
  10. senior_mgmt
  11. recommend
  12. ceo_approv
  13. pros
  14. cons
  15. pros_wc
  16. cons_wc
  17. _year


---
## 3.8 Train/Validation/Test Split

Stratified split to preserve class distribution across train/validation/test sets. Using random seed for reproducibility.

### 3.8.1 Create Binary Target

Create `recommend_binary` (1 = Positive, 0 = Not Positive).

In [16]:
# Create binary target: 1 = Positive (v), 0 = Not Positive (o, x)
df["recommend_binary"] = (df["recommend"] == "v").astype(int)


### 3.8.2 Split

Perform stratified 70/15/15 split.

In [17]:
# Stratified 70/15/15 split
df_train, df_temp = train_test_split(df, test_size=0.30, stratify=df["recommend_binary"], random_state=SEED)
df_val, df_test = train_test_split(df_temp, test_size=0.50, stratify=df_temp["recommend_binary"], random_state=SEED)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# Verify
print("Split sizes and stratification:")
for name, split_df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    pos_pct = split_df["recommend_binary"].mean() * 100
    print(f"  {name}: {len(split_df):,} ({len(split_df)/len(df)*100:.0f}%) — {pos_pct:.1f}% Positive")

Split sizes and stratification:
  Train: 440,112 (70%) — 60.4% Positive
  Val: 94,310 (15%) — 60.4% Positive
  Test: 94,310 (15%) — 60.4% Positive


70/15/15 split, stratified by binary target. Confirmed class distribution preserved across all sets (~60/40 in each).

---
## 3.9 Imputation

Handling the rows with partial sub-rating missingness. Imputer is fit on training data only, then applied to validation and test sets to prevent data leakage.

In [18]:
# --- 3.9.1 Verification: Missing values before imputation ---
print("Missing sub-ratings (Train only — imputation stats computed here):")
for col in CORE_SUBRATINGS:
    n_miss = df_train[col].isna().sum()
    if n_miss > 0:
        print(f"  {col}: {n_miss:,} ({n_miss/len(df_train)*100:.1f}%)")

if df_train[CORE_SUBRATINGS].isna().sum().sum() == 0:
    print("  No missing values — imputation not needed.")

Missing sub-ratings (Train only — imputation stats computed here):
  work_life_balance: 5,153 (1.2%)
  culture_values: 10,902 (2.5%)
  career_opp: 3,467 (0.8%)
  comp_benefits: 5,247 (1.2%)
  senior_mgmt: 8,961 (2.0%)


In [19]:
# --- 3.9.2 Imputation: Median imputation fit on training set only ---

imputer = SimpleImputer(strategy="median")

# Fit on training data only
imputer.fit(df_train[CORE_SUBRATINGS])

# Print learned medians
print("Median values learned from training set:")
for col, med in zip(CORE_SUBRATINGS, imputer.statistics_):
    print(f"  {col:>20s}: {med:.1f}")
print()

# Transform all three splits
df_train[CORE_SUBRATINGS] = imputer.transform(df_train[CORE_SUBRATINGS])
df_val[CORE_SUBRATINGS] = imputer.transform(df_val[CORE_SUBRATINGS])
df_test[CORE_SUBRATINGS] = imputer.transform(df_test[CORE_SUBRATINGS])

print("Imputation applied to train, validation, and test sets.")

Median values learned from training set:
     work_life_balance: 4.0
        culture_values: 4.0
            career_opp: 4.0
         comp_benefits: 4.0
           senior_mgmt: 3.0

Imputation applied to train, validation, and test sets.


In [20]:
# --- 3.9.3 Verify ---
for name, split_df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    n_miss = split_df[CORE_SUBRATINGS].isna().sum().sum()
    print(f"{name}: {'✓ clean' if n_miss == 0 else f'{n_miss} missing'}")

Train: ✓ clean
Val: ✓ clean
Test: ✓ clean


Median imputation for sub-ratings. Fit on training set only to prevent leakage. ~3-4% of rows affected. Verified no missing values remain.

---
## 3.10 Feature Engineering

Creating new features from high-cardinality categorical variables before encoding. Verification uses training set only to inform decisions.

### 3.10.1 Firm Target Encoding

Smoothed target encoding (m=100) preserves firm-level signal while regularising small firms toward the global mean. Computed on training set only to prevent leakage.

In [21]:
# --- 3.10.1 Verification: Confirm EDA pattern holds on training set ---
firm_rates = df_train.groupby("firm")["recommend_binary"].mean()
print(f"Unique firms: {len(firm_rates)}")
print(f"Global recommend rate: {df_train['recommend_binary'].mean()*100:.1f}%")
print(f"Firm rate range (raw): {firm_rates.min()*100:.0f}% - {firm_rates.max()*100:.0f}%")

Unique firms: 428
Global recommend rate: 60.4%
Firm rate range (raw): 0% - 100%


In [22]:
# --- 3.10.1 Apply: Firm target encoding ---
SMOOTHING_M = 100
global_mean = df_train["recommend_binary"].mean()

firm_stats = df_train.groupby("firm")["recommend_binary"].agg(["mean", "count"])
firm_stats["encoded"] = (
    firm_stats["count"] * firm_stats["mean"] + SMOOTHING_M * global_mean
) / (firm_stats["count"] + SMOOTHING_M)

firm_map = firm_stats["encoded"].to_dict()

for name, split in [("train", df_train), ("val", df_val), ("test", df_test)]:
    split["firm_encoded"] = split["firm"].map(firm_map).fillna(global_mean)

print(f"Encoded {len(firm_map)} firms. Range: {firm_stats['encoded'].min():.2f} - {firm_stats['encoded'].max():.2f}")
print(f"Unseen firms: {df_val['firm'].isin(firm_map).eq(False).sum()} val, {df_test['firm'].isin(firm_map).eq(False).sum()} test")

Encoded 428 firms. Range: 0.30 - 0.86
Unseen firms: 0 val, 0 test


In [23]:
# Verification

print(f"Encoded {len(firm_map)} firms. Range: {firm_stats['encoded'].min():.2f} - {firm_stats['encoded'].max():.2f}")
print(f"Applied to train/val/test — unseen firms: {df_val['firm'].isin(firm_map).eq(False).sum()} val, {df_test['firm'].isin(firm_map).eq(False).sum()} test")


Encoded 428 firms. Range: 0.30 - 0.86
Applied to train/val/test — unseen firms: 0 val, 0 test


### 3.10.2 Job Title → Seniority Level

Extract seniority from job titles using keyword matching. Priority order: Management > Senior > Entry. Blank/anonymous → Unknown. Real titles without keywords → Other.

In [ ]:
# --- 3.10.2 Apply + Verify: Job title → seniority level ---

SENIORITY_MAP = [
    ("Management", r"\b(executive|ceo|cfo|cto|coo|chief|president|vp|vice\s*president|director|manager|mgr|lead)\b"),
    ("Senior", r"\b(senior|sr\.?)\b"),
    ("Entry", r"\b(associate|junior|jr\.?|intern)\b"),
]

def extract_seniority(title):
    if pd.isna(title) or str(title).strip() == "":
        return "Unknown"
    t = str(title).lower()
    for level, pattern in SENIORITY_MAP:
        if re.search(pattern, t):
            return level
    return "Unknown" if "anonymous" in t else "Other"

for split in [df_train, df_val, df_test]:
    split["seniority_level"] = split["job_title"].apply(extract_seniority)

# Verify: distribution and recommend rate (compare to EDA: Mgmt/Senior ~66%, Entry ~61%, Unknown ~50-54%, Other ~60%)
print(f"{'Level':<15} | {'Count':>10} | {'% Positive':>10}")
print("-" * 40)
for level in ["Management", "Senior", "Entry", "Unknown", "Other"]:
    subset = df_train[df_train["seniority_level"] == level]
    rate = subset["recommend_binary"].mean() * 100
    print(f"{level:<15} | {len(subset):>10,} | {rate:>9.1f}%")

Level           |      Count | % Positive
----------------------------------------
Management      |     79,229 |      66.3%
Senior          |     31,414 |      64.9%
Entry           |     22,926 |      63.8%
Unknown         |    117,278 |      54.8%
Other           |    189,265 |      60.3%


### 3.10.3 Employee Status (Current / Former)

Map the `current` column to a binary `is_current` feature. 

In [25]:
# --- 3.10.3 Verification: Confirm EDA pattern (Current ~59%, recommend ~56% vs ~44%) ---
raw_current = df_train["current"].str.contains("Current", case=False, na=False).sum()
raw_former = len(df_train) - raw_current
print(f"Current: {raw_current:,} ({raw_current/len(df_train)*100:.1f}%)")
print(f"Former: {raw_former:,} ({raw_former/len(df_train)*100:.1f}%)")

Current: 257,707 (58.6%)
Former: 182,405 (41.4%)


In [26]:
# --- 3.10.3 Apply ---
for split in [df_train, df_val, df_test]:
    split["is_current"] = split["current"].str.contains("Current", case=False, na=False).astype(int)

# Verify: counts match and recommend rates similar to EDA (56% vs 44%)
print(f"is_current=1: {df_train['is_current'].sum():,} — {df_train[df_train['is_current']==1]['recommend_binary'].mean()*100:.1f}% positive")
print(f"is_current=0: {(df_train['is_current']==0).sum():,} — {df_train[df_train['is_current']==0]['recommend_binary'].mean()*100:.1f}% positive")

is_current=1: 257,707 — 66.4% positive
is_current=0: 182,405 — 52.0% positive


### 3.10.4 Review Year

Extract year from `date_review` as categorical. Year range 2012–2021 after section 3.5 filter. One-hot encoded in section 3.11 since EDA showed non-monotonic year effects (peak ~59% in 2012–13, declining to ~44% by 2021).

In [27]:
# --- 3.10.4 Verification: Review year distribution (training set) ---

train_years = pd.to_datetime(df_train["date_review"]).dt.year

print(f"Year range: {train_years.min()} - {train_years.max()}")
print()

print("REVIEW YEAR DISTRIBUTION (training set)\n")

year_dist = train_years.value_counts().sort_index()
year_pct = train_years.value_counts(normalize=True).sort_index() * 100
for year, count in year_dist.items():
    print(f"  {year}: {count:>10,}  ({year_pct[year]:.1f}%)")
print()
print(f"Total reviews: {len(train_years):,}")
print(f"Min year: {train_years.min()}, Max year: {train_years.max()}")

Year range: 2012 - 2021

REVIEW YEAR DISTRIBUTION (training set)

  2012:     12,978  (2.9%)
  2013:     15,153  (3.4%)
  2014:     23,676  (5.4%)
  2015:     40,654  (9.2%)
  2016:     48,613  (11.0%)
  2017:     52,988  (12.0%)
  2018:     49,541  (11.3%)
  2019:     56,123  (12.8%)
  2020:     70,736  (16.1%)
  2021:     69,650  (15.8%)

Total reviews: 440,112
Min year: 2012, Max year: 2021


In [28]:
# --- 3.10.4 Apply + Verify: Extract review year ---
for split in [df_train, df_val, df_test]:
    split["review_year"] = pd.to_datetime(split["date_review"]).dt.year.astype(str)

# Verify: year range and distribution
print(f"Years: {sorted(df_train['review_year'].unique())}")
print(df_train["review_year"].value_counts().sort_index())

Years: ['2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021']
review_year
2012    12978
2013    15153
2014    23676
2015    40654
2016    48613
2017    52988
2018    49541
2019    56123
2020    70736
2021    69650
Name: count, dtype: int64


### 3.10.5 CEO Approval

Map single-letter codes to readable labels (v=Approve, r=Mild, o=No Opinion, x=Disapprove). 87% construct overlap but conceptually distinct from target, will be used in ablation study. One-hot encoded in section 3.11.

In [29]:
# --- 3.10.5 Apply: Rename CEO approval codes ---
CEO_APPROV_MAP = {"v": "Approve", "r": "Mild", "o": "No Opinion", "x": "Disapprove"}

print("Before (training set):")
print(f"  {df_train['ceo_approv'].value_counts().to_dict()}")

for split_df in [df_train, df_val, df_test]:
    split_df["ceo_approv"] = split_df["ceo_approv"].map(CEO_APPROV_MAP)

print("After (training set):")
print(f"  {df_train['ceo_approv'].value_counts().to_dict()}")

Before (training set):
  {'v': 177170, 'o': 117695, 'r': 107491, 'x': 37756}
After (training set):
  {'Approve': 177170, 'No Opinion': 117695, 'Mild': 107491, 'Disapprove': 37756}


---
## 3.11 Encoding

One-hot encode categorical columns (`ceo_approv`, `seniority_level`, `review_year`) with `drop='first'` to avoid multicollinearity. Fit on training data only; `handle_unknown='ignore'` zeros out unseen categories in val/test.

Already numeric (no encoding): `is_current`, sub-ratings, `overall_rating`, `firm_encoded`. Text (`pros`, `cons`): reserved for BERTopic.

In [30]:
# --- 3.11.1 Verification: Columns to encode ---
COLS_TO_ENCODE = ["ceo_approv", "seniority_level", "review_year"]

print("Columns to one-hot encode:")
for col in COLS_TO_ENCODE:
    n_unique = df_train[col].nunique()
    print(f"  {col}: {n_unique} categories -> {n_unique - 1} dummies")

Columns to one-hot encode:
  ceo_approv: 4 categories -> 3 dummies
  seniority_level: 5 categories -> 4 dummies
  review_year: 10 categories -> 9 dummies


In [ ]:
# One-hot encode categorical features ---
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    drop="first",
)
encoder.fit(df_train[COLS_TO_ENCODE])
encoded_cols = encoder.get_feature_names_out(COLS_TO_ENCODE)

for name, split_df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    encoded_array = encoder.transform(split_df[COLS_TO_ENCODE])
    encoded_df = pd.DataFrame(encoded_array, columns=encoded_cols, index=split_df.index)
    split_df.drop(columns=COLS_TO_ENCODE, inplace=True)
    for col in encoded_cols:
        split_df[col] = encoded_df[col].astype(int)

print(f"Encoded {len(COLS_TO_ENCODE)} columns -> {len(encoded_cols)} dummies")
print(f"Total columns: {len(df_train.columns)}")
print("All columns:")
for i, col in enumerate(df_train.columns, 1):
    dtype = df_train[col].dtype
    print(f"  {i:2d}. {col:<40s} ({dtype})")

Encoded 3 columns -> 16 dummies
Total columns: 35
All columns:
   1. firm                                     (str)
   2. date_review                              (str)
   3. job_title                                (str)
   4. current                                  (str)
   5. overall_rating                           (int64)
   6. work_life_balance                        (float64)
   7. culture_values                           (float64)
   8. career_opp                               (float64)
   9. comp_benefits                            (float64)
  10. senior_mgmt                              (float64)
  11. recommend                                (str)
  12. pros                                     (str)
  13. cons                                     (str)
  14. pros_wc                                  (int64)
  15. cons_wc                                  (int64)
  16. _year                                    (int32)
  17. recommend_binary                         (int64)
  18. 

---
## 3.12 Drop Replaced Columns

Raw columns that have been replaced by engineered/encoded features. Dropping from all splits to avoid leakage and reduce noise.

In [32]:
# Drop raw columns
# Leaving firm - will be used in text preprocessing

cols_to_drop = [
    "date_review",   # replaced by review_year_* dummies
    "job_title",     # replaced by seniority_level_* dummies
    "current",       # replaced by is_current
    "recommend",     # replaced by recommend_binary
    "_year",         # intermediate column
    "pros_wc",       # helper from 3.4
    "cons_wc",       # helper from 3.4
]

for split_df in [df_train, df_val, df_test]:
    split_df.drop(columns=cols_to_drop, inplace=True, errors="ignore")

print(f"Final columns: {len(df_train.columns)}")
print(df_train.columns.tolist())

Final columns: 28
['firm', 'overall_rating', 'work_life_balance', 'culture_values', 'career_opp', 'comp_benefits', 'senior_mgmt', 'pros', 'cons', 'recommend_binary', 'firm_encoded', 'is_current', 'ceo_approv_Disapprove', 'ceo_approv_Mild', 'ceo_approv_No Opinion', 'seniority_level_Management', 'seniority_level_Other', 'seniority_level_Senior', 'seniority_level_Unknown', 'review_year_2013', 'review_year_2014', 'review_year_2015', 'review_year_2016', 'review_year_2017', 'review_year_2018', 'review_year_2019', 'review_year_2020', 'review_year_2021']


---
## 3.13 Summary

### Cleaning Pipeline

| Step | Section | Description | Rows After | Removed |
|------|---------|-------------|------------|---------|
| 0 | 3.1 | Raw dataset | 838,566 | -- |
| 1 | 3.2 | Remove exact duplicates | 805,051 | 33,515 |
| 2 | 3.3 | Drop rows with all sub-ratings missing | 668,411 | 136,640 |
| 3 | 3.4 | Remove short text reviews (< 5 words) | 659,607 | 8,804 |
| 4 | 3.5 | Drop pre-2012 (structural missingness) | 628,732 | 30,875 |
| 5 | 3.6 | Drop `outlook` column (leakage) | -- | column only |
| 6 | 3.7 | Drop `location`, `diversity_inclusion`, `headline` | -- | columns only |
| 7 | 3.8 | Create binary target + stratified 70/15/15 split | -- | -- |
| 8 | 3.9 | Median imputation (sub-ratings, ~3-4%) | -- | -- |
| 9 | 3.10 | Feature engineering: target-encode firm, 5-category seniority, is_current, review_year, rename ceo_approv | -- | -- |
| 10 | 3.11 | One-hot encode 3 categoricals (`ceo_approv`, `seniority_level`, `review_year`) | -- | -- |
| 11 | 3.12 | Drop replaced columns (`firm`, `date_review`, `job_title`, `current`, `recommend`) | -- | columns only |

**Note:** Steps 5-11 change columns, not rows.

In [33]:
# Final State 
total_rows = len(df_train) + len(df_val) + len(df_test)
total_removed = STARTING_ROWS - total_rows

print(f"Rows: {STARTING_ROWS:,} -> {total_rows:,} ({total_removed:,} removed, {total_removed/STARTING_ROWS*100:.1f}%)")
print()
print("Splits:")
for name, split_df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    pct = len(split_df) / total_rows * 100
    pos = split_df["recommend_binary"].mean() * 100
    print(f"  {name}: {len(split_df):,} ({pct:.0f}%) — {pos:.1f}% positive")
print()
print(f"Columns: {len(df_train.columns)}")

Rows: 838,566 -> 628,732 (209,834 removed, 25.0%)

Splits:
  Train: 440,112 (70%) — 60.4% positive
  Val: 94,310 (15%) — 60.4% positive
  Test: 94,310 (15%) — 60.4% positive

Columns: 28


In [34]:
# Save Processed Splits

PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Save all columns including text (pros, cons) as-is for BERTopic in next notebook
df_train.to_csv(os.path.join(PROCESSED_DIR, "train.csv"), index=False)
df_val.to_csv(os.path.join(PROCESSED_DIR, "val.csv"), index=False)
df_test.to_csv(os.path.join(PROCESSED_DIR, "test.csv"), index=False)

print(f"Saved to {PROCESSED_DIR}/:")
print(f"  train.csv  ({len(df_train):,} rows)")
print(f"  val.csv    ({len(df_val):,} rows)")
print(f"  test.csv   ({len(df_test):,} rows)")




Saved to ../data/processed/:
  train.csv  (440,112 rows)
  val.csv    (94,310 rows)
  test.csv   (94,310 rows)


In [35]:
# Verify saved files
for fname, expected_df in [("train.csv", df_train), ("val.csv", df_val), ("test.csv", df_test)]:
    reloaded = pd.read_csv(os.path.join(PROCESSED_DIR, fname))
    assert reloaded.shape == expected_df.shape, f"{fname} shape mismatch"
print(f"Saved and verified: train/val/test to {PROCESSED_DIR}")

Saved and verified: train/val/test to ../data/processed
